# Interactive Dashboard Report

## Introduction

*<In two or three sentences: state the question or story your dashboard explores — what can a reader discover with it that they couldn't from a single static chart?>*

**Finished dashboard**

*<Insert a screenshot of the finished dashboard here, and add the link to the live version below.>*

*Live version:* `<paste link to the running dashboard>`

In [0]:
#%pip install dash
import pandas as pd
from dash import dcc, html
import plotly.express as px

## Library choice and fit (15 pts)

**The library: Dash**

- **Created by:** *<who created it — e.g. Plotly, first released in 2017>*
- **License:** *<e.g. MIT — verify and state what it permits>*
- **How to install:** `%pip install dash` (done in the cell above)
- **Declarative or procedural:** *<say which, and justify with how you build UI in Dash (layout as a component tree + callbacks) vs. writing drawing code imperatively>*

**Jupyter integration and limitations**

- *<How Dash runs inside Jupyter — e.g. inline mode / running the dev server, and what the workflow looks like in this notebook>*
- *<Main limitations — e.g. callbacks need a running server, two-process debug workflow, heavier than a pure plotting library for static charts>*

**Why Dash fits this dataset**

*<Why Dash suits your data and questions — e.g. many linked views, filters over a structured tabular dataset, callbacks for cross-filtering.>*

**Alternative considered and rejected:** *<name one (e.g. Streamlit, Panel, Bokeh), and the concrete reason you rejected it>*

## Data

- **Source:** *<where the dataset comes from, with a link>*
- **License:** *<the dataset's license and what it permits>*
- **Size:** *<number of rows × columns, and file size>*

**Columns used**

*<One line per column: name — what it contains and its type.>*

### Cleaning steps

*<Add a code cell under this heading for each cleaning step. Put one sentence above each cell explaining what it does and why it is needed. Example steps: dropping unneeded columns, handling nulls, type conversion, filtering to the time range of interest.>*

In [0]:
#data import and cleaning
df = pd.read_csv('owid_energy.csv')

#data filtering - data prior to 1965 isn't present for countries and is estimated
mask = (
    (df['year'] >= 1965)
    )
    
df = df[mask]

## Visualization techniques (20 pts)

*For each chart type in your dashboard, cover: what it shows, why you used it for this data, and when someone should **not** use it.*

- **<Chart type 1>** — shows…; used here because…; avoid when…
- **<Chart type 2>** — shows…; used here because…; avoid when…
- **<Chart type 3>** — shows…; used here because…; avoid when…

**New-to-class method: <name>**

*<What this method reveals about the data that a chart type covered in class would not.>*

**How the charts complement each other**

*<Why the arrangement works: e.g. overview figures first with detail views linked by shared filters, layout mimics the reader's question flow (filter → compare → drill down).>*

## Building and exploring (35 pts)

Build the dashboard step by step: basic components first, then layout, then linked interactivity. Include a code snippet **and a screenshot** at each key stage.

### Step 1 — Basic components

*<Define individual figures / data preparation for the components. Code snippet + screenshot.>*

In [0]:
def generate_country_data(df: pd.DataFrame(), countries: list = ['World'], years: list = range(1965,2026)) -> pd.DataFrame():
    dfc = df.copy()
    return dfc[dfc['country'].isin(countries) & dfc['year'].isin(years)]


data = generate_country_data(df)

# figure 1
fig1 = px.line(
    data,
    x = 'year',
    y = 'total_energy_per_capita_kwh',
    color = 'country',
    markers = True,
    title = f'Total Energy per Capita by Country ({min(data['year'])}-{max(data['year'])})'
    )

fig1.update_layout(
    xaxis_title = 'Year',
    yaxis_title = 'Total Energy per Capita (kWh)',
    legend_title ='Country',
    hovermode = 'x unified'
)

fig1.show()

In [0]:
def generate_energy_share(df: pd.DataFrame(), countries: list = ['World'], years: list = range(1965,2026)) -> pd.DataFrame():
    dfc = df.copy()

    #define and get columnms we want
    columns = [
        'country',
        'year',
        'bioenergy_electricity_twh',
        'coal_electricity_twh',
        'gas_electricity_twh',
        'hydro_electricity_twh',
        'nuclear_electricity_twh',
        'oil_electricity_twh',
        'solar_electricity_twh',
        'wind_electricity_twh',
        'other_renewables_excluding_bioenergy_electricity_twh',
        ]

    #create dataframe and remove suffixes from energy types
    result = dfc[columns].rename(columns = lambda x: x.removesuffix('_electricity_twh')).rename(columns = {'other_renewables_excluding_bioenergy': 'other renewables'})

    #convert to long format
    result = pd.melt(result, id_vars = ['country', 'year'], var_name = 'energy_type', value_name = 'values')

    #filter to the selected countries and sum their values for each year and energy type
    result = (result[result['country'].isin(countries) & result['year'].isin(years)]
              .groupby(['year', 'energy_type'], as_index = False)['values'].sum())

    #create sort order based on values for final plot year
    order = result[result['year'] == df['year'].max()].sort_values('values', ascending = False)['energy_type'].to_list()

    return result, order

data, order = generate_energy_share(df)

#figure2
fig2 = px.area(
    data,
    x = 'year',
    y = 'values',
    color = 'energy_type',
    line_group = 'energy_type',
    category_orders = {'energy_type': order}, #order of plotting is based on values of right most year being plotted
    title = f'Total Energy Generated by Source ({min(data['year'])}-{max(data['year'])})'
)

fig2.update_layout(
    xaxis_title = 'Year',
    yaxis_title = 'Total Energy Generated by Source (TWh)',
    legend_title ='Energy Source',
    hovermode = 'x unified'
)

fig2.show()

In [0]:
def generate_renewable_energy(df: pd.DataFrame(), years: list = range(1965,2026)) -> pd.DataFrame():
    #copy dataframe and restrict to most recent year
    dfc = df[df['year'] == max(years)].copy()

    #drop columns and na values from iso_code and renewables_electricity_share_pct
    result = dfc[['country','iso_code','year', 'renewables_electricity_share_pct']].dropna(subset = ['iso_code', 'renewables_electricity_share_pct']).rename(columns = {'renewables_electricity_share_pct': 'renewables_share'})

    return result

data = generate_renewable_energy(df)

#figure 3
fig3 = px.choropleth(
    data,
    locations = 'iso_code',
    locationmode = 'ISO-3',
    color = 'renewables_share',
    hover_name = 'country',
    hover_data = {
        'iso_code': False,
        'year': True,
        'renewables_share': ":.1f"
    },
    color_continuous_scale = 'RdYlGn',
    range_color = [0, 100],
    title = f'Renewable Electricity Generation by Country ({max(data['year'])})'
)

fig3.update_layout(
    title_x=0.5,
    height = 650,
    coloraxis_colorbar=dict(
        title="Renewable<br>Generation",
        ticksuffix="%"
    )
)

fig3.show()

In [0]:
def generate_profiles(df: pd.DataFrame(), countries: list = ['World'], years: list = range(1965,2026)) -> pd.DataFrame():
    dfc = df.copy()

    columns = [
        'country',
        'year',
        'electricity_generation_per_capita_kwh',
        'renewables_electricity_share_pct',
        'fossil_fuels_electricity_share_pct',
        'electricity_emissions_mtco2eq',
    ]

    result = (dfc[columns].rename(columns = {
        'electricity_generation_per_capita_kwh': 'energy_per_capita', 
        'renewables_electricity_share_pct': 'renewables_share', 
        'fossil_fuels_electricity_share_pct': 'fossil_fuels_share', 
        'electricity_emissions_mtco2eq': 'emissions'}))

    return result[(result['country'].isin(countries)) & (result['year'] == max(years))].dropna(subset = ['energy_per_capita','renewables_share', 'fossil_fuels_share', 'emissions'])

data = generate_profiles(df, countries = ['United States', 'China','United Kingdom'])
display(data.head())

#figure 4
fig4 = px.parallel_coordinates(
    data,
    dimensions = [
        'energy_per_capita',
        'renewables_share',
        'fossil_fuels_share',
        'emissions'
    ],
    color = 'renewables_share',
    color_continuous_scale = 'RdYlGn',
    range_color = [0, 100],
    labels = {
        'energy_per_capita': 'Energy per capita (kWh)',
        'renewables_share': 'Renewables share (%)',
        'fossil_fuels_share': 'Fossil fuels share (%)',
        'emissions': 'Emissions (MtCO2eq)'
    },
    title = f'Energy Profiles of Countries ({max(data['year'])})'
)

fig4.update_layout(
    title_x=0.5,
    height=650,
    coloraxis_colorbar=dict(
        title="Renewable<br>Share",
        ticksuffix="%"
    )
)

fig4.show()



### Step 2 — Layout

*<Assemble components into the app layout. Code snippet + screenshot of the first full render.>*


In [0]:
# Step 2 — Layout: assemble the four figures into the Dash app
from dash import Dash, dcc, html
from threading import Thread

app = Dash(__name__)

# Row 1 — fig1 (energy per capita) and fig2 (generation by source) side by side
row1 = html.Div(
    [
        dcc.Graph(figure=fig1, style={'flex': 1}),
        dcc.Graph(figure=fig2, style={'flex': 1}),
    ],
    style={'display': 'flex', 'gap': '10px'},
)

# Row 2 — fig3 (renewables choropleth), full width
row2 = html.Div([dcc.Graph(figure=fig3)])

# Row 3 — fig4 (energy profiles), full width
row3 = html.Div([dcc.Graph(figure=fig4)])

app.layout = html.Div([row1, row2, row3])

# Serve the app on a background thread so the notebook stays responsive
Thread(target=lambda: app.run(host='0.0.0.0', port=8050), daemon=True).start()
print('Dashboard serving at http://127.0.0.1:8050')

### Step 3 — Linked interactivity

*<Add callbacks that connect filters/charts. Code snippet + screenshot showing the interaction (before/after states).>*


In [0]:
# Step 3 — Linked interactivity: shared country/year controls drive all four figures
from dash import Dash, dcc, html
from dash.dependencies import Input, Output
from threading import Thread
import plotly.graph_objects as go

def empty_figure(title, message):
    fig = go.Figure()
    fig.update_layout(
        title = title,
        title_x = 0.5,
        annotations = [dict(text = message, showarrow = False, x = 0.5, y = 0.5,
                            xref = 'paper', yref = 'paper', font = dict(size = 16))]
    )
    return fig

# Figure builders — the Step 1 chart code, parameterized by the shared controls
def make_fig1(countries):
    data = generate_country_data(df, countries = countries)
    if data.empty:
        return empty_figure('Total Energy per Capita', 'No data for the selected countries')
    fig = px.line(
        data,
        x = 'year',
        y = 'total_energy_per_capita_kwh',
        color = 'country',
        markers = True,
        title = 'Total Energy per Capita by Country (1965–present)'
    )
    fig.update_layout(
        xaxis_title = 'Year',
        yaxis_title = 'Total Energy per Capita (kWh)',
        legend_title = 'Country',
        hovermode = 'x unified'
    )
    return fig

def make_fig2(countries):
    data, order = generate_energy_share(df, countries = countries)
    if data.empty:
        return empty_figure('Energy Generation by Source', 'No data for the selected countries')
    fig = px.area(
        data,
        x = 'year',
        y = 'values',
        color = 'energy_type',
        line_group = 'energy_type',
        category_orders = {'energy_type': order}
    )
    fig.update_layout(
        xaxis_title = 'Year',
        yaxis_title = 'Total Energy Generated by Source (TWh)',
        legend_title = 'Energy Source',
        hovermode = 'x unified'
    )
    return fig

def make_fig3(year):
    data = generate_renewable_energy(df, years = [year])
    if data.empty:
        return empty_figure(f'Renewable Electricity Generation ({year})', 'No data for this year')
    fig = px.choropleth(
        data,
        locations = 'iso_code',
        locationmode = 'ISO-3',
        color = 'renewables_share',
        hover_name = 'country',
        hover_data = {'iso_code': False, 'year': True, 'renewables_share': ':.1f'},
        color_continuous_scale = 'RdYlGn',
        range_color = [0, 100],
        title = f'Renewable Electricity Generation by Country ({year})'
    )
    fig.update_layout(
        title_x = 0.5,
        height = 650,
        coloraxis_colorbar = dict(title = 'Renewable<br>Generation', ticksuffix = '%')
    )
    return fig

def make_fig4(countries, year):
    data = generate_profiles(df, countries = countries, years = [year])
    if data.empty:
        return empty_figure(f'Energy Profiles ({year})', 'No data for this selection')
    fig = px.parallel_coordinates(
        data,
        dimensions = ['energy_per_capita', 'renewables_share', 'fossil_fuels_share', 'emissions'],
        color = 'renewables_share',
        color_continuous_scale = 'RdYlGn',
        range_color = [0, 100],
        labels = {
            'energy_per_capita': 'Energy per capita (kWh)',
            'renewables_share': 'Renewables share (%)',
            'fossil_fuels_share': 'Fossil fuels share (%)',
            'emissions': 'Emissions (MtCO2eq)'
        },
        title = f'Energy Profiles of Countries ({year})'
    )
    fig.update_layout(
        title_x = 0.5,
        height = 650,
        coloraxis_colorbar = dict(title = 'Renewable<br>Share', ticksuffix = '%')
    )
    return fig

# Shared controls at the top of the dashboard
country_options = sorted(df['country'].unique())
controls = html.Div(
    [
        html.Div(
            [
                html.Label('Countries', htmlFor = 'country-dropdown'),
                dcc.Dropdown(
                    id = 'country-dropdown',
                    options = country_options,
                    value = ['World'],
                    multi = True,
                    placeholder = 'Select countries…'
                ),
            ],
            style = {'flex': 3, 'paddingRight': '20px'}
        ),
        html.Div(
            [
                html.Label('Map & profile year', htmlFor = 'year-slider'),
                dcc.Slider(
                    id = 'year-slider',
                    min = 1965,
                    max = int(df['year'].max()),
                    step = 1,
                    value = int(df['year'].max()),
                    marks = {y: str(y) for y in range(1965, int(df['year'].max()) + 1, 10)}
                ),
            ],
            style = {'flex': 2}
        ),
    ],
    style = {'display': 'flex', 'alignItems': 'flex-end', 'padding': '10px'}
)

app = Dash(__name__)
app.layout = html.Div(
    [
        controls,
        # Row 1 — fig1 and fig2 side by side
        html.Div(
            [dcc.Graph(id = 'fig1-graph'), dcc.Graph(id = 'fig2-graph')],
            style = {'display': 'flex', 'gap': '10px'}
        ),
        # Row 2 — fig3
        html.Div([dcc.Graph(id = 'fig3-graph')]),
        # Row 3 — fig4
        html.Div([dcc.Graph(id = 'fig4-graph')]),
    ]
)

# One callback links the shared controls to all four figures:
# country selection drives fig1, fig2 and fig4; the year slider drives fig3 and fig4.
@app.callback(
    Output('fig1-graph', 'figure'),
    Output('fig2-graph', 'figure'),
    Output('fig3-graph', 'figure'),
    Output('fig4-graph', 'figure'),
    Input('country-dropdown', 'value'),
    Input('year-slider', 'value'),
)
def update_figures(countries, year):
    if not countries:
        countries = ['World']
    return (
        make_fig1(countries),
        make_fig2(countries),
        make_fig3(year),
        make_fig4(countries, year),
    )

# Serve the interactive version on port 8051 (the static Step 2 app is on 8050)
Thread(target = lambda: app.run(host = '0.0.0.0', port = 8051), daemon = True).start()
print('Interactive dashboard serving at http://127.0.0.1:8051')

### Exploration narrative

*<The questions you started with, what you tried, dead ends, and what interaction revealed that a static chart would have missed — e.g. an outlier that turned out to belong to one category, a filter that reversed an initial impression.>*

## AI log and troubleshooting (10 pts)

*<Paste the AI-log template from the assignment here and fill it in: prompts used, what the AI suggested, what you accepted or rejected and why, and problems you hit along the way (e.g. the deprecated `dash_core_components` import and how it was fixed).>*

## Deployment instructions (if applicable)

*<If you hosted the dashboard: step-by-step instructions so a classmate could repeat it — where it runs (e.g. Databricks app / other host), how to start it, and the URL. If you did not deploy it, say so and remove this section.>*